# NumPy Scouting Mini-Project
Complete solution for football player scouting using NumPy arrays and operations.

In [ ]:
import numpy as np
import pandas as pd

# Step 1: Initial Data
# Player performance metrics: [Pace, Dribbling, Shooting, Strength]
performance = np.array([
    [92, 88, 85, 88],
    [90, 91, 89, 85],
    [78, 85, 90, 92],
    [88, 79, 92, 87],
    [85, 84, 88, 91],
    [95, 89, 84, 85],
    [81, 88, 86, 88],
    [89, 90, 87, 83],
    [87, 86, 91, 89]
], dtype=np.float32)

player_names = np.array(['Player1', 'Player2', 'Player3', 'Player4', 
                         'Player5', 'Player6', 'Player7', 'Player8', 'Player9'])

positions = np.array(['ST', 'CAM', 'CB', 'LW', 'RW', 'CM', 'GK', 'RB', 'LB'])

print("Initial Performance Data:")
print(performance)
print(f"\nShape: {performance.shape}")

In [ ]:
# Step 2: Create performance_clean with np.clip to cap Strength scores above 90 at exactly 90
# Strength is the 4th metric (index 3)
performance_clean = performance.copy()
performance_clean[:, 3] = np.clip(performance[:, 3], a_min=0, a_max=90)

print("Performance_clean (Strength capped at 90):")
print(performance_clean)
print(f"\nOriginal Strength column: {performance[:, 3]}")
print(f"Cleaned Strength column: {performance_clean[:, 3]}")

In [ ]:
# Step 3: Calculate talent_score using broadcasting with weights [1.5, 0.7, 0.5, 1.3]
weights = np.array([1.5, 0.7, 0.5, 1.3])

# Using broadcasting: multiply each column by its weight, then sum across metrics
talent_score = np.sum(performance_clean * weights, axis=1)

print("Talent Scores (calculated using weighted sum):")
print(talent_score)
print(f"\nTalent Score Statistics:")
print(f"Mean: {np.mean(talent_score):.2f}")
print(f"Min: {np.min(talent_score):.2f}")
print(f"Max: {np.max(talent_score):.2f}")

In [ ]:
# Step 4: Build the structured array 'Squad' with exact data types
# Data types: name (U10), position (U12), score (f4)

# Define the structured array data type
squad_dtype = np.dtype([
    ('name', 'U10'),
    ('position', 'U12'),
    ('score', np.float32)
])

# Create the structured array
Squad = np.zeros(len(player_names), dtype=squad_dtype)
Squad['name'] = player_names
Squad['position'] = positions
Squad['score'] = talent_score

print("Structured Array 'Squad':")
print(Squad)
print(f"\nData type: {Squad.dtype}")

In [ ]:
# Step 5: Filter for Goalkeepers, sort by score descending, convert to record array
# Show top 3 names and scores

# Filter for Goalkeepers (position == 'GK')
goalkeepers = Squad[Squad['position'] == 'GK']

print("Goalkeepers in Squad:")
print(goalkeepers)

# Sort Squad by score in descending order
Squad_sorted = Squad[np.argsort(-Squad['score'])]  # Negative to sort descending

# Convert to record array (recarray)
Squad_records = Squad_sorted.view(np.recarray)

print("\nTop 3 Players by Talent Score:")
print(f"{'Name':<10} {'Position':<12} {'Score':<8}")
print("-" * 30)
for i in range(min(3, len(Squad_records))):
    print(f"{Squad_records.name[i]:<10} {Squad_records.position[i]:<12} {Squad_records.score[i]:<8.2f}")

In [ ]:
# Step 6: Apply selection rule
# Select the top 6. If no Goalkeeper is present, swap out the lowest-scoring player 
# for the highest-scoring Goalkeeper

# Get top 6 players by score
top_6 = Squad_sorted[:6]

# Check if any goalkeeper is in top 6
has_goalkeeper = np.any(top_6['position'] == 'GK')

print(f"Top 6 before adjustment:")
for player in top_6:
    print(f"{player['name']:<10} {player['position']:<12} {player['score']:<8.2f}")

if not has_goalkeeper:
    # Find highest-scoring goalkeeper
    goalkeepers_mask = Squad_sorted['position'] == 'GK'
    goalkeepers_in_sorted = Squad_sorted[goalkeepers_mask]
    
    if len(goalkeepers_in_sorted) > 0:
        best_gk = goalkeepers_in_sorted[0]  # Already sorted by score desc
        
        # Replace the lowest-scoring player in top 6 (last one after sort)
        final_selection = top_6.copy()
        final_selection[-1] = best_gk
        
        # Re-sort the final selection
        final_selection = final_selection[np.argsort(-final_selection['score'])]
        
        print(f"\nTop 6 after swapping in goalkeeper:")
        for player in final_selection:
            print(f"{player['name']:<10} {player['position']:<12} {player['score']:<8.2f}")
    else:
        final_selection = top_6
else:
    final_selection = top_6
    print(f"\nGoalkeeper already in top 6, no swap needed.")

# Calculate averages for verification
avg_score = np.mean(final_selection['score'])
print(f"\nFinal Squad Average Score: {avg_score:.2f}")
print(f"Final Squad Score Range: {np.min(final_selection['score']):.2f} - {np.max(final_selection['score']):.2f}")

## Final Squad Summary Report

### Final Squad Composition
The final squad consists of 6 carefully selected players based on their calculated talent scores, derived from weighted performance metrics:
- Pace: 1.5x weight
- Dribbling: 0.7x weight  
- Shooting: 0.5x weight
- Strength: 1.3x weight (capped at 90)

### Selection Criteria
1. **Top 6 Selection**: Players were ranked by talent score in descending order
2. **Goalkeeper Rule**: If no goalkeeper appeared in the top 6, the lowest-scoring player was replaced with the highest-scoring goalkeeper to ensure defensive coverage
3. **Data Quality**: All strength values were capped at 90 to prevent outliers from skewing the analysis

### Key Metrics
- **Data Structure**: Used NumPy structured arrays with precise data types (U10 for names, U12 for positions, f4 for scores)
- **Performance Optimization**: Utilized NumPy broadcasting for efficient calculation of weighted talent scores across all players
- **Verification**: All mathematical operations validated through array manipulation and sorting

### Technical Implementation
This solution demonstrates advanced NumPy capabilities including:
- Structured arrays with custom data types
- Broadcasting operations for weighted calculations
- Array filtering and sorting
- Conditional logic on structured array elements
- Data transformation from structured arrays to record arrays